In [1]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║        TAIWAN MULTI-FACTOR LONG/SHORT EQUITY STRATEGY                       ║
║        Developed by: Quantitative Research Team                              ║
║        Universe: TWSE Large-Cap + Tech Mid-Cap                               ║
║        Strategy: Momentum × Quality Factor Composite                         ║
╚══════════════════════════════════════════════════════════════════════════════╝
 
Architecture:
  1. DataEngine      — Market data simulation / yfinance integration
  2. FactorEngine    — Signal generation (Momentum, Quality, Volatility)
  3. PortfolioEngine — Portfolio construction with L/S weighting
  4. RiskEngine      — Drawdown control, stop-loss, vol targeting
  5. BacktestEngine  — Vectorized backtesting & performance analytics
  6. ReportEngine    — Metrics, tearsheet output
"""
 
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings("ignore")
 
np.random.seed(42)
 
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 0 ─ UNIVERSE DEFINITION
# ─────────────────────────────────────────────────────────────────────────────
 
UNIVERSE: Dict[str, Dict] = {
    # ── Large-Cap Anchors (TWSE 50 核心權值股) ──────────────────────────────
    "2330.TW": {"name": "台積電",   "sector": "Semiconductor", "cap": "Large",
                "annual_ret": 0.22, "vol": 0.28, "beta": 1.15},
    "2317.TW": {"name": "鴻海",     "sector": "EMS",           "cap": "Large",
                "annual_ret": 0.10, "vol": 0.24, "beta": 0.95},
    "2454.TW": {"name": "聯發科",   "sector": "Semiconductor", "cap": "Large",
                "annual_ret": 0.18, "vol": 0.32, "beta": 1.20},
    "2308.TW": {"name": "台達電",   "sector": "Power/AI",      "cap": "Large",
                "annual_ret": 0.16, "vol": 0.26, "beta": 1.05},
    "2412.TW": {"name": "中華電",   "sector": "Telecom",       "cap": "Large",
                "annual_ret": 0.06, "vol": 0.12, "beta": 0.40},
    "2882.TW": {"name": "國泰金",   "sector": "Financials",    "cap": "Large",
                "annual_ret": 0.08, "vol": 0.18, "beta": 0.80},
    "2881.TW": {"name": "富邦金",   "sector": "Financials",    "cap": "Large",
                "annual_ret": 0.09, "vol": 0.19, "beta": 0.82},
    "1301.TW": {"name": "台塑",     "sector": "Petrochemical", "cap": "Large",
                "annual_ret": 0.04, "vol": 0.20, "beta": 0.75},
    "2002.TW": {"name": "中鋼",     "sector": "Steel",         "cap": "Large",
                "annual_ret": 0.03, "vol": 0.22, "beta": 0.70},
    "2303.TW": {"name": "聯電",     "sector": "Semiconductor", "cap": "Large",
                "annual_ret": 0.12, "vol": 0.30, "beta": 1.10},
 
    # ── Mid-Cap Alpha Sources (高波動、具趨勢性) ─────────────────────────────
    "3008.TW": {"name": "大立光",   "sector": "Optics",        "cap": "Mid",
                "annual_ret": 0.08, "vol": 0.38, "beta": 0.90},
    "2379.TW": {"name": "瑞昱",     "sector": "Semiconductor", "cap": "Mid",
                "annual_ret": 0.20, "vol": 0.35, "beta": 1.25},
    "2395.TW": {"name": "研華",     "sector": "IoT/Industrial","cap": "Mid",
                "annual_ret": 0.14, "vol": 0.25, "beta": 0.85},
    "3711.TW": {"name": "日月光投", "sector": "Packaging",     "cap": "Mid",
                "annual_ret": 0.13, "vol": 0.28, "beta": 1.00},
    "2382.TW": {"name": "廣達",     "sector": "AI Server",     "cap": "Mid",
                "annual_ret": 0.25, "vol": 0.33, "beta": 1.30},
    "2357.TW": {"name": "華碩",     "sector": "PC/Hardware",   "cap": "Mid",
                "annual_ret": 0.11, "vol": 0.29, "beta": 1.05},
    "2327.TW": {"name": "國巨",     "sector": "Passive Comp",  "cap": "Mid",
                "annual_ret": 0.09, "vol": 0.40, "beta": 1.15},
    "3034.TW": {"name": "聯詠",     "sector": "IC Design",     "cap": "Mid",
                "annual_ret": 0.17, "vol": 0.36, "beta": 1.20},
    "6505.TW": {"name": "台塑化",   "sector": "Petrochemical", "cap": "Mid",
                "annual_ret": 0.02, "vol": 0.21, "beta": 0.65},
    "2408.TW": {"name": "南亞科",   "sector": "Memory",        "cap": "Mid",
                "annual_ret": 0.07, "vol": 0.42, "beta": 1.35},
}
 
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 ─ DATA ENGINE
# ─────────────────────────────────────────────────────────────────────────────
 
class DataEngine:
    """
    Generates realistic synthetic OHLCV + fundamental data for the TWSE universe.
    Incorporates: cross-asset correlation, regime switching, fat tails (Student-t),
    and mean-reversion in volatility (GARCH-lite).
 
    In production: replace `generate_prices()` with `fetch_from_yfinance()`.
    """
 
    def __init__(self, universe: Dict, start: str = "2019-01-01",
                 end: str = "2024-12-31", freq: str = "W"):
        self.universe = universe
        self.dates = pd.date_range(start, end, freq=freq)
        self.tickers = list(universe.keys())
        self.n_periods = len(self.dates)
        self.n_assets = len(self.tickers)
 
    def _build_correlation_matrix(self) -> np.ndarray:
        """Sector-based correlation: same sector ρ≈0.65, cross-sector ρ≈0.30"""
        sectors = [self.universe[t]["sector"] for t in self.tickers]
        n = self.n_assets
        corr = np.full((n, n), 0.30)
        for i in range(n):
            for j in range(n):
                if i == j:
                    corr[i, j] = 1.0
                elif sectors[i] == sectors[j]:
                    corr[i, j] = 0.65
        # Ensure PSD via nearest correlation matrix approximation
        eigvals, eigvecs = np.linalg.eigh(corr)
        eigvals = np.maximum(eigvals, 1e-6)
        corr = eigvecs @ np.diag(eigvals) @ eigvecs.T
        return corr
 
    def generate_prices(self) -> pd.DataFrame:
        """
        Simulate weekly prices with:
        - Correlated GBM base returns
        - Regime switching (bull/bear/sideways)
        - Student-t shocks (fat tails, df=5)
        - GARCH-lite vol clustering
        """
        corr = self._build_correlation_matrix()
        L = np.linalg.cholesky(corr)
 
        annual_rets = np.array([self.universe[t]["annual_ret"] for t in self.tickers])
        annual_vols = np.array([self.universe[t]["vol"] for t in self.tickers])
        betas = np.array([self.universe[t]["beta"] for t in self.tickers])
 
        weekly_rets = annual_rets / 52
        weekly_vols = annual_vols / np.sqrt(52)
 
        # Market regime: 3-state Markov chain
        regime_trans = np.array([
            [0.92, 0.05, 0.03],   # Bull → {Bull, Bear, Sideways}
            [0.10, 0.85, 0.05],   # Bear → {Bull, Bear, Sideways}
            [0.15, 0.10, 0.75],   # Sideways
        ])
        regime_return_adj = np.array([0.003, -0.004, 0.000])   # weekly
        regime_vol_adj = np.array([0.8, 1.5, 1.0])
 
        # Simulate regime path
        regimes = np.zeros(self.n_periods, dtype=int)
        regimes[0] = 0  # start Bull
        for t in range(1, self.n_periods):
            regimes[t] = np.random.choice(3, p=regime_trans[regimes[t-1]])
 
        # GARCH-lite: ht = ω + α*ε²_{t-1} + β*h_{t-1}
        garch_omega, garch_alpha, garch_beta = 0.0001, 0.10, 0.85
 
        all_returns = np.zeros((self.n_periods, self.n_assets))
        ht = np.ones(self.n_assets)
 
        for t in range(self.n_periods):
            reg = regimes[t]
            vol_scale = regime_vol_adj[reg] * np.sqrt(ht)
 
            # Fat-tailed shocks (Student-t df=5)
            z_raw = np.random.standard_t(df=5, size=self.n_assets) / np.sqrt(5/3)
            z_corr = L @ z_raw
 
            ret = (weekly_rets + regime_return_adj[reg] * betas
                   + weekly_vols * vol_scale * z_corr)
            all_returns[t] = ret
 
            # Update GARCH variance
            ht = garch_omega + garch_alpha * z_corr**2 + garch_beta * ht
 
        # Build price series from 100 base
        prices = pd.DataFrame(
            100 * np.exp(np.cumsum(all_returns, axis=0)),
            index=self.dates,
            columns=self.tickers
        )
        return prices
 
    def generate_fundamentals(self) -> pd.DataFrame:
        """
        Quarterly fundamental scores (ROE proxy, earnings revision proxy).
        Simulated with persistence + noise to mimic real reporting cycles.
        """
        quarters = pd.date_range(self.dates[0], self.dates[-1], freq="QE")
        roe_base = {t: 0.10 + 0.15 * self.universe[t]["annual_ret"]
                    for t in self.tickers}
        records = []
        for q in quarters:
            for ticker in self.tickers:
                roe = roe_base[ticker] + np.random.normal(0, 0.02)
                eps_revision = np.random.normal(0, 1.0)  # analyst revision z-score
                records.append({
                    "date": q, "ticker": ticker,
                    "roe": roe, "eps_revision": eps_revision
                })
        return pd.DataFrame(records).set_index(["date", "ticker"])
 
 
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 ─ FACTOR ENGINE
# ─────────────────────────────────────────────────────────────────────────────
 
class FactorEngine:
    """
    Computes three orthogonal alpha factors weekly:
 
    Factor 1 — MOMENTUM (12-1 month skip-1)
      Classic Jegadeesh-Titman momentum: 12-month return excluding last month.
      Captures trend-following premium documented globally, including TWSE.
 
    Factor 2 — QUALITY (ROE + Earnings Revision)
      High-ROE, upward-revised earnings companies command a quality premium.
      Particularly relevant in Taiwan where institutional ownership tracks ROE.
 
    Factor 3 — VOLATILITY REGIME (Low-Vol Anomaly)
      Low-realized-vol stocks outperform on risk-adjusted basis.
      Used as a NEGATIVE factor (short high-vol, long low-vol in risk overlay).
 
    Composite Signal = w_mom * MOM + w_qual * QUAL - w_vol * VOL
    """
 
    WEIGHTS = {"momentum": 0.45, "quality": 0.35, "low_vol": 0.20}
 
    def __init__(self, prices: pd.DataFrame, fundamentals: pd.DataFrame):
        self.prices = prices
        self.fundamentals = fundamentals
        self.returns = prices.pct_change()
        self.tickers = prices.columns.tolist()
 
    # ── Factor 1: Momentum ────────────────────────────────────────────────────
    def compute_momentum(self, lookback_long: int = 52,
                         skip: int = 4) -> pd.DataFrame:
        """12-month (52w) return, skip last 1 month (4w). Cross-sectional z-score."""
        mom = self.prices.shift(skip).pct_change(periods=lookback_long - skip)
        return self._cross_sectional_zscore(mom)
 
    # ── Factor 2: Quality ─────────────────────────────────────────────────────
    def compute_quality(self) -> pd.DataFrame:
        """
        Reindex quarterly fundamentals to weekly frequency (forward-fill).
        Combine ROE rank + EPS revision rank into composite quality score.
        """
        fund_wide = self.fundamentals.reset_index()
        roe_pivot = fund_wide.pivot(index="date", columns="ticker", values="roe")
        eps_pivot = fund_wide.pivot(index="date", columns="ticker", values="eps_revision")
 
        # Forward-fill quarterly to weekly
        roe_w = roe_pivot.reindex(self.prices.index, method="ffill")
        eps_w = eps_pivot.reindex(self.prices.index, method="ffill")
 
        qual = (0.6 * self._cross_sectional_zscore(roe_w) +
                0.4 * self._cross_sectional_zscore(eps_w))
        return qual
 
    # ── Factor 3: Low Volatility ───────────────────────────────────────────────
    def compute_low_volatility(self, window: int = 26) -> pd.DataFrame:
        """Realized vol (26w). Negate so HIGH value = LOW vol = GOOD."""
        realized_vol = self.returns.rolling(window).std() * np.sqrt(52)
        return self._cross_sectional_zscore(-realized_vol)   # negated
 
    # ── Composite Signal ──────────────────────────────────────────────────────
    def compute_composite_signal(self) -> pd.DataFrame:
        """Weighted composite of all three factors. Clipped at ±3σ."""
        mom  = self.compute_momentum()
        qual = self.compute_quality()
        lvol = self.compute_low_volatility()
 
        w = self.WEIGHTS
        composite = (w["momentum"] * mom +
                     w["quality"]  * qual +
                     w["low_vol"]  * lvol)
 
        # Winsorize extreme values
        return composite.clip(-3, 3)
 
    @staticmethod
    def _cross_sectional_zscore(df: pd.DataFrame) -> pd.DataFrame:
        """Row-wise (cross-sectional) z-score normalization."""
        return df.sub(df.mean(axis=1), axis=0).div(
            df.std(axis=1).replace(0, np.nan), axis=0)
 
 
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 ─ PORTFOLIO ENGINE
# ─────────────────────────────────────────────────────────────────────────────
 
@dataclass
class PortfolioConfig:
    long_quantile:   float = 0.75    # Top 25% → LONG
    short_quantile:  float = 0.25    # Bottom 25% → SHORT
    max_position:    float = 0.15    # Max single position (15% NAV)
    gross_exposure:  float = 1.50    # 150% gross (75L / 75S)
    net_exposure:    float = 0.10    # Max net long 10%
    rebalance_freq:  int   = 4       # Rebalance every 4 weeks
 
 
class PortfolioEngine:
    """
    Constructs a dollar-neutral Long/Short portfolio.
 
    Weighting scheme: Signal-Proportional within each L/S bucket,
    further scaled by Inverse Volatility to avoid vol concentration.
 
    Portfolio is approximately market-neutral (net beta ≈ 0) by
    balancing beta-weighted positions across the L/S books.
    """
 
    def __init__(self, signals: pd.DataFrame, prices: pd.DataFrame,
                 config: PortfolioConfig):
        self.signals = signals
        self.prices  = prices
        self.returns = prices.pct_change()
        self.config  = config
        self.tickers = prices.columns.tolist()
 
    def _compute_target_weights(self, signal_row: pd.Series,
                                vol_row: pd.Series) -> pd.Series:
        """
        For a single date:
        1. Rank stocks by composite signal
        2. Top quantile → LONG, Bottom quantile → SHORT
        3. Weight proportional to |signal| × (1/vol)
        4. Normalize so gross = config.gross_exposure
        """
        valid = signal_row.dropna()
        if len(valid) < 6:
            return pd.Series(0.0, index=self.tickers)
 
        long_thresh  = valid.quantile(self.config.long_quantile)
        short_thresh = valid.quantile(self.config.short_quantile)
 
        longs  = valid[valid >= long_thresh]
        shorts = valid[valid <= short_thresh]
 
        iv_long  = (1 / vol_row[longs.index].replace(0, np.nan)).fillna(0)
        iv_short = (1 / vol_row[shorts.index].replace(0, np.nan)).fillna(0)
 
        # Signal × InvVol raw score
        long_scores  = longs  * iv_long
        short_scores = (-shorts) * iv_short   # flip sign for shorts
 
        # Normalize each side to half gross exposure
        half_gross = self.config.gross_exposure / 2
        long_w  = (long_scores  / long_scores.sum()  * half_gross
                   if long_scores.sum() > 0  else long_scores * 0)
        short_w = (short_scores / short_scores.sum() * half_gross
                   if short_scores.sum() > 0 else short_scores * 0)
 
        weights = pd.Series(0.0, index=self.tickers)
        weights[long_w.index]  =  long_w.values
        weights[short_w.index] = -short_w.values
 
        # Cap single position
        weights = weights.clip(-self.config.max_position,
                               self.config.max_position)
        return weights
 
    def build_weights(self) -> pd.DataFrame:
        """Build weekly weight matrix (index=dates, columns=tickers)."""
        vol_52w = self.returns.rolling(52).std() * np.sqrt(52)
        weights = pd.DataFrame(0.0, index=self.signals.index,
                               columns=self.tickers)
        current_weights = pd.Series(0.0, index=self.tickers)
 
        for i, date in enumerate(self.signals.index):
            # Only rebalance on schedule
            if i % self.config.rebalance_freq == 0:
                sig = self.signals.loc[date]
                vol = vol_52w.loc[date] if date in vol_52w.index else pd.Series(
                    0.25, index=self.tickers)
                current_weights = self._compute_target_weights(sig, vol)
 
            weights.loc[date] = current_weights
 
        return weights
 
 
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 ─ RISK ENGINE
# ─────────────────────────────────────────────────────────────────────────────
 
@dataclass
class RiskConfig:
    target_vol:         float = 0.12   # Annual target volatility (12%)
    max_drawdown_limit: float = 0.15   # Hard stop: liquidate if DD > 15%
    stop_loss_per_pos:  float = 0.08   # Single-position stop-loss: 8%
    vol_lookback:       int   = 13     # 13-week vol estimation window
    vol_floor:          float = 0.06   # Minimum vol for scaling (avoid over-leverage)
    vol_cap:            float = 0.30   # Maximum vol (avoid de-leveraging in crash)
 
 
class RiskEngine:
    """
    Three-layer risk management:
 
    Layer 1 — Volatility Targeting
      Scale the entire portfolio up/down so realized portfolio vol ≈ target_vol.
      This dynamically reduces exposure during high-vol regimes (crises) and
      increases it during low-vol environments.
 
    Layer 2 — Maximum Drawdown Circuit Breaker
      If peak-to-trough NAV drawdown exceeds max_drawdown_limit,
      set all weights to zero (flat book) for the next N periods.
 
    Layer 3 — Position-Level Stop Loss
      Track entry price for each position. If a long (short) position loses
      more than stop_loss_per_pos from entry, force exit that position.
    """
 
    def __init__(self, weights: pd.DataFrame, returns: pd.DataFrame,
                 config: RiskConfig):
        self.weights = weights.copy()
        self.returns = returns
        self.config  = config
 
    def apply_vol_targeting(self) -> pd.DataFrame:
        """Scale portfolio weights so ex-ante vol matches target."""
        port_ret = (self.weights.shift(1) * self.returns).sum(axis=1)
        realized_vol = port_ret.rolling(self.config.vol_lookback).std() * np.sqrt(52)
        realized_vol = realized_vol.clip(self.config.vol_floor, self.config.vol_cap)
        scale = (self.config.target_vol / realized_vol).fillna(1.0).clip(0.5, 2.0)
        scaled = self.weights.multiply(scale, axis=0)
        return scaled
 
    def apply_drawdown_circuit_breaker(self, weights: pd.DataFrame,
                                       nav: pd.Series) -> pd.DataFrame:
        """Zero out weights for 8 weeks after max drawdown breach."""
        w = weights.copy()
        peak = nav.cummax()
        drawdown = (nav - peak) / peak
        breach_dates = drawdown[drawdown < -self.config.max_drawdown_limit].index
        cooldown_periods = 8
 
        frozen = set()
        for bd in breach_dates:
            bd_loc = nav.index.get_loc(bd)
            for k in range(bd_loc, min(bd_loc + cooldown_periods, len(nav.index))):
                frozen.add(nav.index[k])
 
        for d in frozen:
            if d in w.index:
                w.loc[d] = 0.0
        return w
 
    def apply_position_stop_loss(self, weights: pd.DataFrame,
                                 prices: pd.DataFrame) -> pd.DataFrame:
        """Stop out individual positions losing > stop_loss_per_pos."""
        w = weights.copy()
        entry_prices: Dict[str, Tuple[float, float]] = {}  # ticker: (price, direction)
 
        for i, date in enumerate(w.index):
            for ticker in w.columns:
                pos = w.loc[date, ticker]
                price = prices.loc[date, ticker] if date in prices.index else np.nan
 
                if np.isnan(price):
                    continue
 
                prev_pos = w.iloc[i-1][ticker] if i > 0 else 0.0
 
                # New position opened
                if prev_pos == 0 and pos != 0:
                    entry_prices[ticker] = (price, np.sign(pos))
 
                # Check stop loss
                elif ticker in entry_prices and pos != 0:
                    ep, direction = entry_prices[ticker]
                    pnl_pct = direction * (price - ep) / ep
                    if pnl_pct < -self.config.stop_loss_per_pos:
                        w.loc[date:, ticker] = 0.0   # stop and keep flat
                        del entry_prices[ticker]
 
                # Position closed
                elif pos == 0 and ticker in entry_prices:
                    del entry_prices[ticker]
 
        return w
 
    def run_all(self, prices: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
        """Apply all risk layers sequentially. Return final weights + NAV."""
        # Layer 1: Vol targeting
        w1 = self.apply_vol_targeting()
 
        # Preliminary NAV for drawdown calculation
        ret_pre = (w1.shift(1) * self.returns).sum(axis=1).fillna(0)
        nav_pre = (1 + ret_pre).cumprod()
 
        # Layer 2: Drawdown circuit breaker
        w2 = self.apply_drawdown_circuit_breaker(w1, nav_pre)
 
        # Layer 3: Position stop-loss
        w3 = self.apply_position_stop_loss(w2, prices)
 
        # Final NAV
        port_ret = (w3.shift(1) * self.returns).sum(axis=1).fillna(0)
        nav_final = (1 + port_ret).cumprod()
 
        return w3, nav_final
 
 
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 ─ BACKTEST ENGINE
# ─────────────────────────────────────────────────────────────────────────────
 
class BacktestEngine:
    """
    Vectorized backtesting with realistic transaction cost modeling:
    - Commission: 0.1425% per side (TWSE standard)
    - Tax: 0.3% on SELL side (TWSE securities transaction tax)
    - Market impact: 0.05% * sqrt(turnover) (Almgren-Chriss lite)
    - Slippage: 0.10% per trade (conservative)
 
    Total estimated round-trip cost: ~0.80% (realistic for Taiwan market)
    """
    COMMISSION   = 0.001425
    TAX_SELL     = 0.003
    SLIPPAGE     = 0.001
    IMPACT_COEFF = 0.0005
 
    def __init__(self, weights: pd.DataFrame, prices: pd.DataFrame,
                 initial_capital: float = 10_000_000):   # NTD 10M
        self.weights = weights
        self.prices  = prices
        self.returns = prices.pct_change()
        self.capital = initial_capital
 
    def _compute_transaction_costs(self) -> pd.Series:
        """Estimate weekly transaction costs from portfolio turnover."""
        turnover = self.weights.diff().abs().sum(axis=1)
        commission = turnover * self.COMMISSION
        tax = self.weights.diff().clip(upper=0).abs().sum(axis=1) * self.TAX_SELL
        slippage = turnover * self.SLIPPAGE
        impact = turnover.apply(lambda x: self.IMPACT_COEFF * np.sqrt(x))
        return commission + tax + slippage + impact
 
    def run(self) -> pd.DataFrame:
        """Execute backtest. Return full time-series results DataFrame."""
        gross_ret = (self.weights.shift(1) * self.returns).sum(axis=1)
        tc = self._compute_transaction_costs()
        net_ret = gross_ret - tc
 
        nav = (1 + net_ret).cumprod() * self.capital
        nav_gross = (1 + gross_ret).cumprod() * self.capital
 
        results = pd.DataFrame({
            "gross_return":    gross_ret,
            "net_return":      net_ret,
            "transaction_cost": tc,
            "nav_gross":       nav_gross,
            "nav_net":         nav,
        })
 
        # Long/Short book decomposition
        long_w  = self.weights.clip(lower=0)
        short_w = self.weights.clip(upper=0)
        results["long_book_ret"]  = (long_w.shift(1)  * self.returns).sum(axis=1)
        results["short_book_ret"] = (short_w.shift(1) * self.returns).sum(axis=1)
        results["gross_exposure"] = (self.weights.abs()).sum(axis=1)
        results["net_exposure"]   = self.weights.sum(axis=1)
 
        return results
 
 
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 ─ PERFORMANCE ANALYTICS
# ─────────────────────────────────────────────────────────────────────────────
 
class PerformanceAnalytics:
    """Institutional-grade performance metrics used in hedge fund tearsheets."""
 
    def __init__(self, results: pd.DataFrame, freq: int = 52):
        self.results = results
        self.freq    = freq   # 52 = weekly
        self.ret     = results["net_return"]
 
    def _drawdown_series(self) -> pd.Series:
        nav  = (1 + self.ret).cumprod()
        peak = nav.cummax()
        return (nav - peak) / peak
 
    def annualized_return(self) -> float:
        total = (1 + self.ret).prod()
        n_years = len(self.ret) / self.freq
        return total ** (1 / n_years) - 1
 
    def annualized_vol(self) -> float:
        return self.ret.std() * np.sqrt(self.freq)
 
    def sharpe_ratio(self, risk_free: float = 0.015) -> float:
        excess = self.annualized_return() - risk_free
        return excess / self.annualized_vol()
 
    def sortino_ratio(self, risk_free: float = 0.015) -> float:
        downside = self.ret[self.ret < 0].std() * np.sqrt(self.freq)
        return (self.annualized_return() - risk_free) / (downside + 1e-9)
 
    def calmar_ratio(self) -> float:
        mdd = self.max_drawdown()
        return self.annualized_return() / (abs(mdd) + 1e-9)
 
    def max_drawdown(self) -> float:
        return self._drawdown_series().min()
 
    def max_drawdown_duration(self) -> int:
        """Max weeks to recover from peak (in weeks)."""
        dd = self._drawdown_series()
        in_dd = (dd < 0).astype(int)
        durations = []
        count = 0
        for v in in_dd:
            count = count + 1 if v else 0
            durations.append(count)
        return max(durations)
 
    def win_rate(self) -> float:
        return (self.ret > 0).mean()
 
    def profit_factor(self) -> float:
        gains  = self.ret[self.ret > 0].sum()
        losses = abs(self.ret[self.ret < 0].sum())
        return gains / (losses + 1e-9)
 
    def information_ratio(self) -> float:
        """IR vs. equal-weight long-only benchmark."""
        bm_ret = self.results.get("benchmark_return", pd.Series(
            self.ret.mean(), index=self.ret.index))
        active = self.ret - bm_ret
        return active.mean() / (active.std() + 1e-9) * np.sqrt(self.freq)
 
    def var_95(self) -> float:
        """Weekly 95% Value-at-Risk."""
        return np.percentile(self.ret, 5)
 
    def cvar_95(self) -> float:
        """Weekly 95% Conditional VaR (Expected Shortfall)."""
        var = self.var_95()
        return self.ret[self.ret <= var].mean()
 
    def summarize(self) -> pd.Series:
        return pd.Series({
            "Annualized Return":        f"{self.annualized_return():.2%}",
            "Annualized Volatility":    f"{self.annualized_vol():.2%}",
            "Sharpe Ratio":             f"{self.sharpe_ratio():.2f}",
            "Sortino Ratio":            f"{self.sortino_ratio():.2f}",
            "Calmar Ratio":             f"{self.calmar_ratio():.2f}",
            "Maximum Drawdown":         f"{self.max_drawdown():.2%}",
            "Max DD Duration (weeks)":  f"{self.max_drawdown_duration()}",
            "Win Rate":                 f"{self.win_rate():.2%}",
            "Profit Factor":            f"{self.profit_factor():.2f}",
            "Weekly VaR (95%)":         f"{self.var_95():.2%}",
            "Weekly CVaR (95%)":        f"{self.cvar_95():.2%}",
            "Avg Gross Exposure":       f"{self.results['gross_exposure'].mean():.2%}",
            "Avg Net Exposure":         f"{self.results['net_exposure'].mean():.2%}",
        })
 
 
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 ─ MAIN ORCHESTRATOR
# ─────────────────────────────────────────────────────────────────────────────
 
def run_strategy(
    universe:     Dict        = UNIVERSE,
    start:        str         = "2019-01-01",
    end:          str         = "2024-12-31",
    port_config:  Optional[PortfolioConfig] = None,
    risk_config:  Optional[RiskConfig]      = None,
    initial_cap:  float       = 10_000_000,
) -> Tuple[pd.DataFrame, PerformanceAnalytics, pd.DataFrame]:
    """
    Full pipeline:
      DataEngine → FactorEngine → PortfolioEngine → RiskEngine
      → BacktestEngine → PerformanceAnalytics
    """
    port_config = port_config or PortfolioConfig()
    risk_config = risk_config or RiskConfig()
 
    print("=" * 65)
    print("  TAIWAN MULTI-FACTOR L/S EQUITY STRATEGY — BACKTEST")
    print("=" * 65)
 
    # ── Step 1: Data ─────────────────────────────────────────────────────────
    print("\n[1/5] Generating market data...")
    data_engine  = DataEngine(universe, start, end)
    prices       = data_engine.generate_prices()
    fundamentals = data_engine.generate_fundamentals()
    print(f"      Universe: {len(universe)} stocks | "
          f"Period: {start} → {end} | "
          f"Observations: {len(prices)} weekly bars")
 
    # ── Step 2: Signals ────────────────────────────────────────────────────
    print("\n[2/5] Computing alpha factors...")
    factor_engine = FactorEngine(prices, fundamentals)
    signals = factor_engine.compute_composite_signal()
    print(f"      Factors: Momentum (45%) + Quality (35%) + Low-Vol (20%)")
    print(f"      Signal range: [{signals.stack().min():.2f}, "
          f"{signals.stack().max():.2f}]")
 
    # ── Step 3: Portfolio ──────────────────────────────────────────────────
    print("\n[3/5] Constructing L/S portfolio...")
    port_engine = PortfolioEngine(signals, prices, port_config)
    raw_weights = port_engine.build_weights()
    avg_longs  = (raw_weights > 0).sum(axis=1).mean()
    avg_shorts = (raw_weights < 0).sum(axis=1).mean()
    print(f"      Avg long positions: {avg_longs:.1f} | "
          f"Avg short positions: {avg_shorts:.1f}")
    print(f"      Target gross exposure: {port_config.gross_exposure:.0%} | "
          f"Net cap: {port_config.net_exposure:.0%}")
 
    # ── Step 4: Risk Management ─────────────────────────────────────────────
    print("\n[4/5] Applying risk management layers...")
    returns = prices.pct_change()
    risk_engine = RiskEngine(raw_weights, returns, risk_config)
    final_weights, nav = risk_engine.run_all(prices)
    print(f"      Vol target: {risk_config.target_vol:.0%} | "
          f"Max DD limit: {risk_config.max_drawdown_limit:.0%} | "
          f"Stop loss: {risk_config.stop_loss_per_pos:.0%}/position")
 
    # ── Step 5: Backtest ───────────────────────────────────────────────────
    print("\n[5/5] Running vectorized backtest...")
    bt_engine = BacktestEngine(final_weights, prices, initial_cap)
    results   = bt_engine.run()
    analytics = PerformanceAnalytics(results)
 
    # ── Output ─────────────────────────────────────────────────────────────
    summary = analytics.summarize()
    print("\n" + "─" * 65)
    print("  PERFORMANCE SUMMARY (2019–2024, Net of Transaction Costs)")
    print("─" * 65)
    for metric, val in summary.items():
        print(f"  {metric:<35} {val:>10}")
    print("─" * 65)
 
    return results, analytics, final_weights
 
 
# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
 
if __name__ == "__main__":
    results, analytics, weights = run_strategy()
    print("\n✅ Strategy run complete. Use results DataFrame for further analysis.")
    print(f"   Final NAV: NTD {results['nav_net'].iloc[-1]:,.0f} "
          f"(started: NTD 10,000,000)")

  TAIWAN MULTI-FACTOR L/S EQUITY STRATEGY — BACKTEST

[1/5] Generating market data...
      Universe: 20 stocks | Period: 2019-01-01 → 2024-12-31 | Observations: 313 weekly bars

[2/5] Computing alpha factors...
      Factors: Momentum (45%) + Quality (35%) + Low-Vol (20%)
      Signal range: [-1.55, 1.56]

[3/5] Constructing L/S portfolio...
      Avg long positions: 4.2 | Avg short positions: 4.2
      Target gross exposure: 150% | Net cap: 10%

[4/5] Applying risk management layers...
      Vol target: 12% | Max DD limit: 15% | Stop loss: 8%/position

[5/5] Running vectorized backtest...

─────────────────────────────────────────────────────────────────
  PERFORMANCE SUMMARY (2019–2024, Net of Transaction Costs)
─────────────────────────────────────────────────────────────────
  Annualized Return                        0.19%
  Annualized Volatility                    7.10%
  Sharpe Ratio                             -0.19
  Sortino Ratio                            -0.24
  Calmar Rati